# Этап 2: качество данных

Вход: `data/raw/merged_price_m2.csv`. Целевой признак для распределения: `label` (≈ цена за м²).

Части: **1)** детект и визуализации → **2)** два прогона `fix` (balanced vs conservative) → **3)** вывод.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.data_quality_agent import DataQualityAgent, load_strategy

df = pd.read_csv(ROOT / "data/raw/merged_price_m2.csv", low_memory=False)
agent = DataQualityAgent()
report = agent.detect_issues(df)
report.keys()

In [ ]:
# Пропуски (топ колонок)
miss = pd.Series(report["missing"]["per_column"]).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 4))
sns.barplot(x=miss.values, y=miss.index)
plt.title("Пропуски по колонкам (топ-15)")
plt.tight_layout()
plt.show()

In [ ]:
# Распределение price_per_m2 по источнику
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="source", y="price_per_m2")
plt.xticks(rotation=15, ha="right")
plt.title("Цена за м² по источнику (сырьё)")
plt.tight_layout()
plt.show()

In [ ]:
# Вариант A: balanced (clip IQR)
str_bal = load_strategy(ROOT / "strategy.yaml")
agent.label_column = str_bal.get("label_column", "label")
agent.iqr_multiplier = float(str_bal.get("iqr_multiplier", 1.5))
df_bal = agent.fix(df.copy(), str_bal)
cmp_bal = agent.compare(df, df_bal)
cmp_bal

In [ ]:
# Вариант B: conservative (удаление выбросов по IQR)
str_con = load_strategy(ROOT / "strategy_conservative.yaml")
agent.iqr_multiplier = float(str_con.get("iqr_multiplier", 1.5))
df_con = agent.fix(df.copy(), str_con)
cmp_con = agent.compare(df, df_con)
cmp_con

## Обоснование

- **balanced** (`strategy.yaml`): подрезает экстремальные значения по IQR без потери строк — удобно, когда важен объём обучающей выборки и регрессия устойчива к остаточному шуму.
- **conservative** (`strategy_conservative.yaml`): удаляет строки-выбросы — меньше экстремальных точек, но сильнее сдвигается распределение и падает `n`.

Для пайплайна по умолчанию сохранён **balanced**; итоговый файл: `data/processed/merged_clean.csv` (см. `python quality_step.py`).